# Notebook 02 — Create Larger Dataset

## NYC Yellow Taxi Trip Data — January to March 2022

This notebook creates a larger dataset by combining multiple months of NYC Yellow Taxi data.

The assignment requires experiments with datasets of different sizes.  
Until now, the project contains:

- a small dataset with 100,000 rows
- a base dataset using January 2022

This notebook creates a larger dataset by combining:

- January 2022
- February 2022
- March 2022

The output will be saved in the `data/processed` directory and later used in the benchmark notebook.

In [1]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import time
import gc
import warnings

warnings.filterwarnings("ignore")

## 1. Project paths

The raw monthly files are stored in `data/raw`.

The combined and cleaned larger dataset will be saved in `data/processed`.

In [2]:
PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw directory:", RAW_DIR)
print("Processed directory:", PROCESSED_DIR)
print("Results directory:", RESULTS_DIR)

Project root: /Users/nunoantunes/Desktop/nyc_taxi_project
Raw directory: /Users/nunoantunes/Desktop/nyc_taxi_project/data/raw
Processed directory: /Users/nunoantunes/Desktop/nyc_taxi_project/data/processed
Results directory: /Users/nunoantunes/Desktop/nyc_taxi_project/results


## 2. Raw files

The larger dataset will be created from three monthly Parquet files.

In [3]:
monthly_files = [
    RAW_DIR / "yellow_tripdata_2022-01.parquet",
    RAW_DIR / "yellow_tripdata_2022-02.parquet",
    RAW_DIR / "yellow_tripdata_2022-03.parquet",
]

for file_path in monthly_files:
    if file_path.exists():
        size_mb = file_path.stat().st_size / (1024 ** 2)
        print(f"Found: {file_path.name} | Size: {size_mb:.2f} MB")
    else:
        raise FileNotFoundError(f"Missing file: {file_path}")

Found: yellow_tripdata_2022-01.parquet | Size: 36.37 MB
Found: yellow_tripdata_2022-02.parquet | Size: 43.50 MB
Found: yellow_tripdata_2022-03.parquet | Size: 53.10 MB


## 3. Cleaning function

The same cleaning strategy from Notebook 01 is reused here.

This keeps the preprocessing consistent across the base dataset and the larger dataset.

In [4]:
def clean_taxi_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean NYC Yellow Taxi data using simple and explainable rules.
    """

    df = df.copy()

    # Normalize column names
    df.columns = [col.strip() for col in df.columns]

    # Remove missing target values
    df = df[df["fare_amount"].notna()]

    # Keep reasonable fare amounts
    df = df[df["fare_amount"] > 0]
    df = df[df["fare_amount"] < 300]

    # Keep reasonable trip distances
    df = df[df["trip_distance"].notna()]
    df = df[df["trip_distance"] > 0]
    df = df[df["trip_distance"] < 200]

    # Passenger count cleaning
    if "passenger_count" in df.columns:
        df = df[df["passenger_count"].notna()]
        df = df[df["passenger_count"] > 0]
        df = df[df["passenger_count"] <= 8]

    # Trip duration
    df["trip_duration_minutes"] = (
        df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
    ).dt.total_seconds() / 60

    df = df[df["trip_duration_minutes"].notna()]
    df = df[df["trip_duration_minutes"] > 0]
    df = df[df["trip_duration_minutes"] <= 24 * 60]

    # Time-based features
    df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
    df["pickup_dayofweek"] = df["tpep_pickup_datetime"].dt.dayofweek
    df["pickup_month"] = df["tpep_pickup_datetime"].dt.month

    return df

## 4. Load, clean and combine monthly files

Each month is loaded and cleaned separately.

This avoids keeping unnecessary raw rows in memory for longer than needed.

In [5]:
cleaned_months = []
monthly_summary = []

start_total = time.perf_counter()

for file_path in monthly_files:
    print("=" * 80)
    print(f"Processing file: {file_path.name}")

    start_month = time.perf_counter()

    df_month_raw = pd.read_parquet(file_path)
    raw_rows = len(df_month_raw)
    raw_columns = df_month_raw.shape[1]

    print(f"Raw shape: {df_month_raw.shape}")

    df_month_clean = clean_taxi_data(df_month_raw)
    clean_rows = len(df_month_clean)
    clean_columns = df_month_clean.shape[1]

    print(f"Clean shape: {df_month_clean.shape}")

    cleaned_months.append(df_month_clean)

    end_month = time.perf_counter()

    monthly_summary.append({
        "file": file_path.name,
        "raw_rows": raw_rows,
        "raw_columns": raw_columns,
        "clean_rows": clean_rows,
        "clean_columns": clean_columns,
        "removed_rows": raw_rows - clean_rows,
        "removed_percentage": round((raw_rows - clean_rows) / raw_rows * 100, 2),
        "processing_time_seconds": round(end_month - start_month, 4)
    })

    del df_month_raw
    gc.collect()

end_total = time.perf_counter()

print("=" * 80)
print(f"Total processing time: {end_total - start_total:.4f} seconds")

Processing file: yellow_tripdata_2022-01.parquet
Raw shape: (2463931, 19)
Clean shape: (2301788, 23)
Processing file: yellow_tripdata_2022-02.parquet
Raw shape: (2979431, 19)
Clean shape: (2771838, 23)
Processing file: yellow_tripdata_2022-03.parquet
Raw shape: (3627882, 19)
Clean shape: (3380328, 23)
Total processing time: 4.9678 seconds


In [6]:
monthly_summary_df = pd.DataFrame(monthly_summary)
monthly_summary_df

,file,raw_rows,raw_columns,clean_rows,clean_columns,removed_rows,removed_percentage,processing_time_seconds
0,yellow_tripdata_2022-01.parquet,2463931,19,2301788,23,162143,6.58,1.7362
1,yellow_tripdata_2022-02.parquet,2979431,19,2771838,23,207593,6.97,1.4258
2,yellow_tripdata_2022-03.parquet,3627882,19,3380328,23,247554,6.82,1.7324


## 5. Combine months

The cleaned monthly DataFrames are concatenated into one larger dataset.

In [7]:
start_time = time.perf_counter()

df_large = pd.concat(cleaned_months, ignore_index=True)

end_time = time.perf_counter()

print("Large dataset created.")
print(f"Concatenation time: {end_time - start_time:.4f} seconds")
print(f"Large dataset shape: {df_large.shape[0]:,} rows x {df_large.shape[1]} columns")

Large dataset created.
Concatenation time: 0.4354 seconds
Large dataset shape: 8,453,954 rows x 23 columns


In [8]:
df_large.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,trip_duration_minutes,pickup_hour,pickup_dayofweek,pickup_month
0,1,2022-01-01 00:35:40,2022-01-01 00:53:29,2.0,3.80,1.0,N,142,236,1,...,3.65,0.0,0.3,21.95,2.5,0.0,17.816667,0,5,1
1,1,2022-01-01 00:33:43,2022-01-01 00:42:07,1.0,2.10,1.0,N,236,42,1,...,4.00,0.0,0.3,13.30,0.0,0.0,8.400000,0,5,1
2,2,2022-01-01 00:53:21,2022-01-01 01:02:19,1.0,0.97,1.0,N,166,166,1,...,1.76,0.0,0.3,10.56,0.0,0.0,8.966667,0,5,1
3,2,2022-01-01 00:25:21,2022-01-01 00:35:23,1.0,1.09,1.0,N,114,68,2,...,0.00,0.0,0.3,11.80,2.5,0.0,10.033333,0,5,1
4,2,2022-01-01 00:36:48,2022-01-01 01:14:20,1.0,4.30,1.0,N,68,163,1,...,3.00,0.0,0.3,30.30,2.5,0.0,37.533333,0,5,1


In [9]:
df_large.dtypes

VendorID                          int64
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int64
DOLocationID                      int64
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
airport_fee                     float64
trip_duration_minutes           float64
pickup_hour                       int32
pickup_dayofweek                  int32
pickup_month                      int32
dtype: object

## 6. Create machine learning version

A machine learning version of the larger dataset is also created using the same selected columns from Notebook 01.

In [10]:
selected_columns = [
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "RatecodeID",
    "trip_duration_minutes",
    "pickup_hour",
    "pickup_dayofweek",
    "pickup_month",
    "fare_amount"
]

available_columns = [col for col in selected_columns if col in df_large.columns]

df_large_ml = df_large[available_columns].dropna()

print("Selected columns:")
print(available_columns)
print()
print(f"Large ML dataset shape: {df_large_ml.shape}")

Selected columns:
['passenger_count', 'trip_distance', 'PULocationID', 'DOLocationID', 'payment_type', 'RatecodeID', 'trip_duration_minutes', 'pickup_hour', 'pickup_dayofweek', 'pickup_month', 'fare_amount']

Large ML dataset shape: (8453954, 11)


## 7. Dataset comparison

This table compares the small, base and larger datasets used in the project.

In [11]:
small_path = PROCESSED_DIR / "yellow_tripdata_2022-01_small_100k.parquet"
base_path = PROCESSED_DIR / "yellow_tripdata_2022-01_clean.parquet"

dataset_comparison = []

if small_path.exists():
    small_rows = len(pd.read_parquet(small_path, columns=["fare_amount"]))
    dataset_comparison.append({
        "dataset": "taxi_small_100k",
        "description": "Sample from January 2022",
        "rows": small_rows,
        "source_files": "yellow_tripdata_2022-01.parquet"
    })

if base_path.exists():
    base_rows = len(pd.read_parquet(base_path, columns=["fare_amount"]))
    dataset_comparison.append({
        "dataset": "taxi_base_jan_2022",
        "description": "Cleaned January 2022",
        "rows": base_rows,
        "source_files": "yellow_tripdata_2022-01.parquet"
    })

dataset_comparison.append({
    "dataset": "taxi_large_q1_2022",
    "description": "Cleaned January, February and March 2022",
    "rows": len(df_large),
    "source_files": "yellow_tripdata_2022-01/02/03.parquet"
})

dataset_comparison_df = pd.DataFrame(dataset_comparison)
dataset_comparison_df

,dataset,description,rows,source_files
0,taxi_small_100k,Sample from January 2022,100000,yellow_tripdata_2022-01.parquet
1,taxi_base_jan_2022,Cleaned January 2022,2301788,yellow_tripdata_2022-01.parquet
2,taxi_large_q1_2022,"Cleaned January, February and March 2022",8453954,yellow_tripdata_2022-01/02/03.parquet


## 8. Save larger datasets

The larger datasets are saved in Parquet format.

The outputs are:

- full larger cleaned dataset
- larger machine learning dataset

In [12]:
large_clean_path = PROCESSED_DIR / "yellow_tripdata_2022_q1_clean.parquet"
large_ml_path = PROCESSED_DIR / "yellow_tripdata_2022_q1_ml.parquet"

start_time = time.perf_counter()

df_large.to_parquet(large_clean_path, index=False)
df_large_ml.to_parquet(large_ml_path, index=False)

end_time = time.perf_counter()

print("Large datasets saved successfully.")
print(f"Saving time: {end_time - start_time:.4f} seconds")
print()
print(large_clean_path)
print(large_ml_path)

Large datasets saved successfully.
Saving time: 2.8853 seconds

/Users/nunoantunes/Desktop/nyc_taxi_project/data/processed/yellow_tripdata_2022_q1_clean.parquet
/Users/nunoantunes/Desktop/nyc_taxi_project/data/processed/yellow_tripdata_2022_q1_ml.parquet


In [13]:
for path in [large_clean_path, large_ml_path]:
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"{path.name}: {size_mb:.2f} MB")

yellow_tripdata_2022_q1_clean.parquet: 174.64 MB
yellow_tripdata_2022_q1_ml.parquet: 57.90 MB


## 9. Save metadata

Metadata and summaries are saved to the `results` directory.

These files will be useful for the final report.

In [14]:
monthly_summary_path = RESULTS_DIR / "large_dataset_monthly_summary.csv"
dataset_comparison_path = RESULTS_DIR / "dataset_size_comparison.csv"

monthly_summary_df.to_csv(monthly_summary_path, index=False)
dataset_comparison_df.to_csv(dataset_comparison_path, index=False)

print("Saved metadata files:")
print(monthly_summary_path)
print(dataset_comparison_path)

Saved metadata files:
/Users/nunoantunes/Desktop/nyc_taxi_project/results/large_dataset_monthly_summary.csv
/Users/nunoantunes/Desktop/nyc_taxi_project/results/dataset_size_comparison.csv


## 10. Final notes

The larger dataset was successfully created by combining three months of NYC Yellow Taxi data.

This dataset will be used in the benchmark notebook to evaluate how the selected libraries behave as the data size increases.

In [16]:
print("Notebook 02 completed successfully.")
print("Large dataset saved at:", large_clean_path)
print("Large ML dataset saved at:", large_ml_path)

Notebook 02 completed successfully.
Large dataset saved at: /Users/nunoantunes/Desktop/nyc_taxi_project/data/processed/yellow_tripdata_2022_q1_clean.parquet
Large ML dataset saved at: /Users/nunoantunes/Desktop/nyc_taxi_project/data/processed/yellow_tripdata_2022_q1_ml.parquet
